# Reconstruction impact report

This report reads only the published CSV/JSON files copied into `inputs/`. It does not load H5AD, pickle checkpoints, or rerun scientific calculations.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import re

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

SPEC = json.loads(Path("inputs.json").read_text(encoding="utf-8"))
FIGURES = Path("figures")
FIGURES.mkdir(parents=True, exist_ok=True)
FIGURE_SUMMARY = []


def slug(value):
    text = str(value)
    prefix = re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("._")[:20] or "task"
    return f"{prefix}_{sha256(text.encode()).hexdigest()[:10]}"


def value_label(value):
    try:
        if pd.isna(value):
            return "null"
    except (TypeError, ValueError):
        pass
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def load_table(task, role):
    item = task.get("artifacts", {}).get(role)
    if not item:
        return None
    return pd.read_csv(Path(item["path"]))


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / name, dpi=160, bbox_inches="tight")
    plt.close(fig)


def empty_figure(title, message, name):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.axis("off")
    ax.set_title(title)
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True)
    save(fig, name)


def record_figure(kind, task_id, filters, *, n_rows=0, n_plotted=0, status="computed", values=None):
    record = {
        "kind": kind,
        "task_id": str(task_id),
        "n_rows": int(n_rows),
        "n_plotted": int(n_plotted),
        "status": status,
    }
    record.update({key: value_label(value) for key, value in filters.items()})
    if values is not None:
        record["values"] = [value_label(value) for value in values]
    FIGURE_SUMMARY.append(record)


def table_ids(*frames):
    values = set()
    for frame in frames:
        if frame is not None and "comparison_id" in frame.columns:
            values.update(frame["comparison_id"].dropna().astype(str))
    return sorted(values)


def first_label(frame, column, default=""):
    if column not in frame.columns or frame.empty:
        return default
    return value_label(frame[column].iloc[0])


for task in SPEC["tasks"]:
    task_slug = slug(task["task_id"])
    title = task["task_id"]
    status = task.get("latest_status", "unknown")
    stale = task.get("stale", False)
    suffix = "; published result is stale" if stale else ""
    display(Markdown(f"## {title}\n\nLatest task status: **{status}**{suffix}."))

    summary = load_table(task, "partition_summary")
    contingency = load_table(task, "partition_contingency")
    partition_ids = table_ids(summary, contingency)
    if summary is not None and "comparison_id" in summary.columns:
        for comparison_id, frame in summary.groupby("comparison_id", sort=True):
            display(Markdown(f"### Partition summary: {comparison_id}"))
            display(frame.head(30))
    elif summary is not None:
        display(Markdown("### Partition summary"))
        display(summary.head(30))

    required_contingency = {
        "comparison_id", "comparison_edge", "normalization",
        "raw_cluster", "recon_cluster", "value",
    }
    if contingency is None or not required_contingency.issubset(contingency.columns):
        for comparison_id in partition_ids or ["unavailable"]:
            empty_figure(
                f"Partition contingency: {comparison_id}",
                "Published contingency table is unsupported or unavailable",
                f"{task_slug}_partition_contingency_{slug(comparison_id)}.png",
            )
            record_figure(
                "partition_contingency",
                task["task_id"],
                {"comparison_id": comparison_id},
                status="unsupported",
            )
    else:
        absolute = contingency.loc[contingency["normalization"].eq("absolute")]
        for comparison_id in partition_ids or ["unavailable"]:
            subset = absolute.loc[absolute["comparison_id"].astype(str).eq(str(comparison_id))]
            if subset.empty:
                empty_figure(
                    f"Partition contingency: {comparison_id}",
                    "No published absolute contingency rows",
                    f"{task_slug}_partition_contingency_{slug(comparison_id)}.png",
                )
                record_figure(
                    "partition_contingency",
                    task["task_id"],
                    {"comparison_id": comparison_id},
                    status="unsupported",
                )
                continue
            for comparison_edge, frame in subset.groupby("comparison_edge", sort=True, dropna=False):
                matrix = frame.pivot_table(
                    index="raw_cluster",
                    columns="recon_cluster",
                    values="value",
                    aggfunc="sum",
                    fill_value=0,
                )
                scope = first_label(frame, "scope", "unknown_scope")
                edge = value_label(comparison_edge)
                figure_title = f"Partition contingency: {scope}\n{edge}"
                fig, ax = plt.subplots(figsize=(5.5, 4.2))
                image = ax.imshow(matrix.to_numpy(), cmap="Blues")
                ax.set(
                    xticks=range(matrix.shape[1]),
                    xticklabels=matrix.columns,
                    yticks=range(matrix.shape[0]),
                    yticklabels=matrix.index,
                    xlabel="Reconstructed cluster",
                    ylabel="Raw cluster",
                    title=figure_title,
                )
                fig.colorbar(image, ax=ax, fraction=.046, pad=.04)
                save(
                    fig,
                    f"{task_slug}_partition_contingency_{slug(comparison_id)}_{slug(edge)}.png",
                )
                record_figure(
                    "partition_contingency",
                    task["task_id"],
                    {"comparison_id": comparison_id, "comparison_edge": edge, "scope": scope},
                    n_rows=len(frame),
                    n_plotted=int(matrix.size),
                    values=[frame["value"].sum()],
                )

    windows = load_table(task, "spatial_window_metrics")
    required_windows = {
        "comparison_id", "metric", "baseline", "scale", "status",
        "window_x", "window_y", "delta",
    }
    if windows is None or not required_windows.issubset(windows.columns):
        empty_figure(
            "Spatial windows",
            "Published spatial window table is unsupported or unavailable",
            f"{task_slug}_spatial_windows.png",
        )
        record_figure(
            "spatial_windows",
            task["task_id"],
            {},
            status="unsupported",
        )
    else:
        window_keys = ["comparison_id", "metric", "baseline", "scale"]
        for key_values, frame in windows.groupby(window_keys, sort=True, dropna=False):
            filters = dict(zip(window_keys, key_values))
            suffix = "_".join(slug(value_label(filters[key])) for key in window_keys)
            figure_title = (
                f"Spatial windows: {first_label(frame, 'scope', 'unknown_scope')}\n"
                f"{value_label(filters['metric'])} / {value_label(filters['baseline'])} / "
                f"scale={value_label(filters['scale'])}"
            )
            computed = frame.loc[frame["status"].eq("computed")].copy()
            if computed.empty:
                states = ", ".join(sorted(computed["status"].astype(str).unique()))
                if not states:
                    states = ", ".join(sorted(frame["status"].astype(str).unique()))
                empty_figure(
                    figure_title,
                    f"No computed rows; published status: {states}",
                    f"{task_slug}_spatial_windows_{suffix}.png",
                )
                record_figure(
                    "spatial_windows",
                    task["task_id"],
                    filters,
                    n_rows=len(frame),
                    status="unsupported",
                )
                continue
            x = pd.to_numeric(computed["window_x"], errors="coerce")
            y = pd.to_numeric(computed["window_y"], errors="coerce")
            delta = pd.to_numeric(computed["delta"], errors="coerce")
            if x.isna().any() or y.isna().any() or delta.isna().any():
                empty_figure(
                    figure_title,
                    "Computed rows have invalid plotting values",
                    f"{task_slug}_spatial_windows_{suffix}.png",
                )
                record_figure(
                    "spatial_windows",
                    task["task_id"],
                    filters,
                    n_rows=len(frame),
                    status="unsupported",
                )
                continue
            fig, ax = plt.subplots(figsize=(5.5, 4.2))
            image = ax.scatter(x, y, c=delta, cmap="coolwarm", s=45)
            ax.set(xlabel="window x", ylabel="window y", title=figure_title)
            fig.colorbar(image, ax=ax, fraction=.046, pad=.04, label="reconstruction - baseline")
            save(fig, f"{task_slug}_spatial_windows_{suffix}.png")
            record_figure(
                "spatial_windows",
                task["task_id"],
                filters,
                n_rows=len(frame),
                n_plotted=len(computed),
                values=sorted(delta.tolist()),
            )

    extent = load_table(task, "spatial_region_extent_by_anatomy")
    required_extent = {"comparison_id", "level1_region", "region_type", "region_area_um2"}
    if extent is None or not required_extent.issubset(extent.columns):
        empty_figure(
            "State/Gain extent",
            "Published region extent table is unsupported or unavailable",
            f"{task_slug}_anatomy_extent.png",
        )
        record_figure(
            "anatomy_extent",
            task["task_id"],
            {},
            status="unsupported",
        )
    else:
        for comparison_id, frame in extent.groupby("comparison_id", sort=True, dropna=False):
            comparison_label = value_label(comparison_id)
            labels = frame["level1_region"].astype(str) + " / " + frame["region_type"].astype(str)
            values = pd.to_numeric(frame["region_area_um2"], errors="coerce")
            figure_title = f"State/Gain extent: {first_label(frame, 'scope', 'unknown_scope')}"
            figure_name = f"{task_slug}_anatomy_extent_{slug(comparison_label)}.png"
            if values.isna().all():
                empty_figure(figure_title, "No measured region area", figure_name)
                record_figure(
                    "anatomy_extent",
                    task["task_id"],
                    {"comparison_id": comparison_label},
                    n_rows=len(frame),
                    status="unsupported",
                )
                continue
            fig, ax = plt.subplots(figsize=(7, 4.0))
            ax.bar(labels, values)
            ax.set(title=figure_title, ylabel="region_area_um2")
            ax.tick_params(axis="x", rotation=45)
            save(fig, figure_name)
            record_figure(
                "anatomy_extent",
                task["task_id"],
                {"comparison_id": comparison_label},
                n_rows=len(frame),
                n_plotted=len(frame),
                values=values.tolist(),
            )

    anatomy = load_table(task, "anatomy_summary")
    if anatomy is not None:
        if "comparison_id" in anatomy.columns:
            for comparison_id, frame in anatomy.groupby("comparison_id", sort=True):
                display(Markdown(f"### Anatomy stratification: {comparison_id}"))
                display(frame.head(30))
        else:
            display(Markdown("### Anatomy stratification"))
            display(anatomy.head(30))

    assignments = load_table(task, "raw_level2_assignments")
    required_level2 = {"comparison_id", "raw_level2"}
    if assignments is None or not required_level2.issubset(assignments.columns):
        empty_figure(
            "Raw Level2 comparison",
            "Published Raw Level2 table is unsupported or unavailable",
            f"{task_slug}_raw_level2.png",
        )
        record_figure(
            "raw_level2",
            task["task_id"],
            {},
            status="unsupported",
        )
    else:
        for comparison_id, frame in assignments.groupby("comparison_id", sort=True, dropna=False):
            comparison_label = value_label(comparison_id)
            display(Markdown(f"### Raw Level2 comparison: {comparison_label}"))
            display(frame.head(30))
            counts = frame["raw_level2"].astype(str).value_counts().sort_index()
            fig, ax = plt.subplots(figsize=(6.5, 3.8))
            ax.bar(counts.index, counts.to_numpy())
            ax.set(
                title=f"Raw Level2 assignments: {first_label(frame, 'scope', 'unknown_scope')}",
                xlabel="Raw Level2",
                ylabel="units",
            )
            ax.tick_params(axis="x", rotation=45)
            save(fig, f"{task_slug}_raw_level2_{slug(comparison_label)}.png")
            record_figure(
                "raw_level2",
                task["task_id"],
                {"comparison_id": comparison_label},
                n_rows=len(frame),
                n_plotted=len(frame),
                values=counts.tolist(),
            )

print("FIGURE_SUMMARY_JSON=" + json.dumps(FIGURE_SUMMARY, sort_keys=True, separators=(",", ":")))
